# 12 — Capstone: Reliability Monitor

End-to-end reliability system combining semantic entropy, hallucination detection, conformal coverage, and routing.

In [ ]:
# Reliability monitor pipeline
import numpy as np

np.random.seed(42)

class ReliabilityMonitor:
    """
    Production reliability pipeline for LLM outputs.
    Components:
      1. Semantic entropy (multi-sample uncertainty)
      2. SelfCheckGPT consistency
      3. Conformal coverage threshold
      4. Routing decision
    """

    def __init__(self, abstain_threshold=0.4, route_threshold=0.6):
        self.abstain_threshold = abstain_threshold
        self.route_threshold = route_threshold
        self.n_answered = 0
        self.n_abstained = 0
        self.n_routed = 0

    def semantic_entropy(self, answers):
        """Simplified: cluster by word overlap."""
        clusters = []
        for a in answers:
            words_a = set(a.lower().split())
            placed = False
            for cluster in clusters:
                words_c = set(cluster[0].lower().split())
                if len(words_a & words_c) / max(len(words_a | words_c), 1) > 0.4:
                    cluster.append(a)
                    placed = True
                    break
            if not placed:
                clusters.append([a])
        probs = [len(c)/len(answers) for c in clusters]
        return -sum(p*np.log(p+1e-10) for p in probs)

    def self_check_score(self, main, samples):
        """Average sentence consistency across samples."""
        main_words = set(main.lower().split())
        scores = []
        for s in samples:
            s_words = set(s.lower().split())
            overlap = len(main_words & s_words) / max(len(main_words | s_words), 1)
            scores.append(overlap)
        return np.mean(scores) if scores else 0

    def assess(self, question, main_answer, sample_answers):
        """
        Returns: decision (answer/abstain/route), confidence, metrics.
        """
        se = self.semantic_entropy(sample_answers)
        sc = self.self_check_score(main_answer, sample_answers)

        # Composite confidence: high consistency + low entropy = high confidence
        se_norm = np.exp(-se)  # 1=certain, 0=maximally uncertain
        composite_conf = (se_norm + sc) / 2

        if composite_conf < self.abstain_threshold:
            decision = 'ABSTAIN'
            self.n_abstained += 1
        elif composite_conf < self.route_threshold:
            decision = 'ROUTE_LARGE'
            self.n_routed += 1
        else:
            decision = 'ANSWER'
            self.n_answered += 1

        return {
            'decision': decision,
            'confidence': composite_conf,
            'semantic_entropy': se,
            'self_check_score': sc,
        }

monitor = ReliabilityMonitor(abstain_threshold=0.35, route_threshold=0.6)

# Test scenarios
scenarios = [
    {
        'q': 'What is the capital of France?',
        'main': 'Paris is the capital of France',
        'samples': ['Paris capital of France', 'The capital is Paris', 'Paris France capital',
                    'Paris is France capital city', 'Capital of France: Paris'],
    },
    {
        'q': 'When was the Eiffel Tower built?',
        'main': 'The Eiffel Tower was built in 1887',
        'samples': ['Built in 1889 for the world fair', 'Completed 1889', 'Around 1887-1889',
                    'The tower dates to the late 1880s', 'Some sources say 1889'],
    },
    {
        'q': 'What caused the 2008 financial crisis?',
        'main': 'Subprime mortgage collapse triggered a global recession',
        'samples': ['Housing bubble burst', 'Bank failures from risky lending',
                    'Regulatory failures allowed excessive risk', 'Multiple complex factors',
                    'Lehman Brothers collapse was the trigger'],
    },
]

print('=== Reliability Monitor Results ===')
for sc_data in scenarios:
    result = monitor.assess(sc_data['q'], sc_data['main'], sc_data['samples'])
    print(f'\nQ: "{sc_data["q"]}')
    print(f'  Decision: {result["decision"]}')
    print(f'  Confidence: {result["confidence"]:.3f}')
    print(f'  Semantic entropy: {result["semantic_entropy"]:.3f}')
    print(f'  Self-check: {result["self_check_score"]:.3f}')

In [ ]:
# Process 100 simulated queries and report monitor statistics
import matplotlib.pyplot as plt

def make_query(q_id):
    np.random.seed(q_id)
    difficulty = np.random.uniform(0, 1)
    base_words = ['capital', 'city', 'country', 'year', 'person', 'built', 'created']
    main = ' '.join(np.random.choice(base_words, 5))
    # High difficulty: diverse samples
    vocab = ['answer', 'result', 'fact', 'information', 'source', 'evidence', 'data']
    n_unique = max(1, int((1 - difficulty) * 7))
    options = [' '.join(np.random.choice(base_words, 4)) for _ in range(n_unique)]
    samples = np.random.choice(options, 5, replace=True).tolist()
    return main, samples

monitor2 = ReliabilityMonitor(abstain_threshold=0.35, route_threshold=0.6)
conf_scores = []
decisions = []

for q_id in range(100):
    main, samples = make_query(q_id)
    result = monitor2.assess(f'q{q_id}', main, samples)
    conf_scores.append(result['confidence'])
    decisions.append(result['decision'])

decision_counts = {d: decisions.count(d) for d in set(decisions)}
print('Decision distribution over 100 queries:')
for d, c in sorted(decision_counts.items()):
    print(f'  {d}: {c} ({c:.0f}%)')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(conf_scores, bins=20, color='steelblue', alpha=0.8)
ax1.axvline(monitor2.abstain_threshold, color='tomato', label=f'Abstain threshold={monitor2.abstain_threshold}')
ax1.axvline(monitor2.route_threshold, color='green', label=f'Route threshold={monitor2.route_threshold}')
ax1.set_xlabel('Confidence score')
ax1.set_ylabel('Count')
ax1.set_title('Confidence Distribution')
ax1.legend()

ax2.bar(list(decision_counts.keys()), list(decision_counts.values()),
         color=['tomato', 'orange', 'steelblue'])
ax2.set_xlabel('Decision')
ax2.set_ylabel('Count')
ax2.set_title('Decision Distribution')

plt.suptitle('LLM Reliability Monitor')
plt.tight_layout()
plt.savefig('/tmp/reliability_monitor.png', dpi=100, bbox_inches='tight')
plt.show()
print('\nSeries 25 (LLM Uncertainty & Reliability) complete.')